# LAB 07 — TRANSFORMER ARCHITECTURE
**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Chủ đề**: From Attention to Transformer  
**Sinh viên thực hiện**: Nguyễn Văn Thành  


In [ ]:
import os
import re
import time
import math
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)


## 1. Tiền Xử Lý Dữ Liệu IMDb Movie Reviews
Phân chia tập dữ liệu chuẩn: 70% Train, 15% Validation, 15% Test. Xây dựng từ điển kích thước 10,000 từ vựng.


In [ ]:
dataset_file = '../Lab5/dataset/IMDB Dataset.csv'
if not os.path.exists(dataset_file):
    dataset_file = 'd:/Hus/NLP/Lab5/dataset/IMDB Dataset.csv'

dataframe = pd.read_csv(dataset_file)
dataframe['clean_text'] = dataframe['review'].apply(lambda s: re.sub(r'<[^>]+>', ' ', s).lower().strip())
dataframe['label'] = dataframe['sentiment'].map({'positive': 1, 'negative': 0})

sample_dataframe = dataframe.sample(n=10000, random_state=42).reset_index(drop=True)
train_df, test_df = train_test_split(sample_dataframe, test_size=0.3, random_state=42, stratify=sample_dataframe['label'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['label'])

token_counts = Counter()
for text in train_df['clean_text']:
    token_counts.update(re.findall(r'\b\w+\b', text))

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in token_counts.most_common(9998):
    vocab[word] = len(vocab)

def tokenize_and_pad(text, max_len=100):
    words = re.findall(r'\b\w+\b', text.lower())
    indices = [vocab.get(w, vocab['<UNK>']) for w in words[:max_len]]
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    return indices

class IMDbDataset(Dataset):
    def __init__(self, texts, labels, max_len=100):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        seq = tokenize_and_pad(self.texts[idx], self.max_len)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

train_loader = DataLoader(IMDbDataset(train_df['clean_text'], train_df['label'], max_len=100), batch_size=64, shuffle=True)
val_loader = DataLoader(IMDbDataset(val_df['clean_text'], val_df['label'], max_len=100), batch_size=64, shuffle=False)
test_loader = DataLoader(IMDbDataset(test_df['clean_text'], test_df['label'], max_len=100), batch_size=64, shuffle=False)


## 2. Experiment 1: Positional Encoding & Permutation Test
Khảo sát ma trận mã hóa vị trí hàm Sin/Cos và kiểm chứng thực nghiệm tính bất biến hoán vị của Self-Attention.


In [ ]:
def compute_positional_encoding_numpy(seq_len, d_model):
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    position = np.arange(0, seq_len, dtype=np.float32)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2, dtype=np.float32) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

pe_matrix = compute_positional_encoding_numpy(100, 128)
plt.figure(figsize=(9, 5))
plt.imshow(pe_matrix, aspect='auto', cmap='viridis', origin='lower')
plt.colorbar(label='Encoding Value')
plt.xlabel('Embedding Dimension (0 to 127)')
plt.ylabel('Token Position (0 to 99)')
plt.title('Sinusoidal Positional Encoding Matrix (T=100, d_model=128)')
plt.tight_layout()
plt.savefig('positional_encoding.png', dpi=300)
plt.show()


### Kiểm nghiệm Permutation trên 2 câu hoán vị từ
So sánh $S_1 = \text{"dog bites man"}$ và $S_2 = \text{"man bites dog"}$ khi CÓ và KHÔNG CÓ Positional Encoding.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        q_proj = self.w_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k_proj = self.w_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v_proj = self.w_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(q_proj, k_proj.transpose(-2, -1)) / self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        dropped_weights = self.dropout(attention_weights)

        context = torch.matmul(dropped_weights, v_proj)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        output = self.w_o(context)
        return output, attention_weights

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attention_weights = self.attention(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x, attention_weights

emb_perm = nn.Embedding(len(vocab), 128)
block_perm = TransformerEncoderBlock(d_model=128, n_heads=4, d_ff=512)
pe_perm = SinusoidalPositionalEncoding(d_model=128)

s1_tokens = ["dog", "bites", "man"]
s2_tokens = ["man", "bites", "dog"]

idx1 = torch.tensor([[vocab.get(w, vocab['<UNK>']) for w in s1_tokens]], dtype=torch.long)
idx2 = torch.tensor([[vocab.get(w, vocab['<UNK>']) for w in s2_tokens]], dtype=torch.long)

with torch.no_grad():
    e1 = emb_perm(idx1)
    e2 = emb_perm(idx2)
    
    out1_no_pe, _ = block_perm(e1)
    out2_no_pe, _ = block_perm(e2)
    pool1_no_pe = out1_no_pe.mean(dim=1)
    pool2_no_pe = out2_no_pe.mean(dim=1)
    dist_no_pe = torch.norm(pool1_no_pe - pool2_no_pe).item()
    
    out1_pe, _ = block_perm(pe_perm(e1))
    out2_pe, _ = block_perm(pe_perm(e2))
    pool1_pe = out1_pe.mean(dim=1)
    pool2_pe = out2_pe.mean(dim=1)
    dist_pe = torch.norm(pool1_pe - pool2_pe).item()

print(f"Permutation L2 Distance WITHOUT Positional Encoding: {dist_no_pe:.8f}")
print(f"Permutation L2 Distance WITH Positional Encoding: {dist_pe:.8f}")


## 3. Shape Verification Qua Khối Transformer Block (Section 18)
Kiểm tra kích thước Tensor từ đầu vào $(B=8, T=100, d_{model}=128, n_{heads}=4, d_k=32, d_{ff}=512)$.


In [ ]:
x_dummy = torch.randn(8, 100, 128)
block_test = TransformerEncoderBlock(d_model=128, n_heads=4, d_ff=512)
out_dummy, attn_dummy = block_test(x_dummy)

print("Input Tensor Shape:", list(x_dummy.shape))
print("Block Output Shape:", list(out_dummy.shape))
print("Attention Weights Shape:", list(attn_dummy.shape))


## 4. Định Nghĩa Mô Hình Phân Loại Transformer & LSTM
Xây dựng lớp phân loại chuỗi hoàn chỉnh với các tùy chọn Pooling: Mean Pooling và First-Token (CLS-style) Pooling.


In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=512, pooling='mean', dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.pooling = pooling
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=1000)
        self.transformer_block = TransformerEncoderBlock(d_model, n_heads, d_ff, dropout=dropout)
        self.fc = nn.Linear(d_model, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        emb = self.embedding(x) * math.sqrt(self.d_model)
        x_pe = self.pos_encoder(emb)
        out, attn_weights = self.transformer_block(x_pe)
        if self.pooling == 'mean':
            pooled = out.mean(dim=1)
        elif self.pooling == 'first':
            pooled = out[:, 0, :]
        logits = self.fc(pooled).squeeze(-1)
        return self.sigmoid(logits), attn_weights

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.lstm = nn.LSTM(d_model, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        logits = self.fc(h_n[-1]).squeeze(-1)
        return self.sigmoid(logits)

def train_and_eval(model, train_ld, val_ld, test_ld, epochs=5, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    
    start_time = time.time()
    for ep in range(epochs):
        model.train()
        running_train_loss = 0.0
        for x_b, y_b in train_ld:
            optimizer.zero_grad()
            out = model(x_b)
            probs = out[0] if isinstance(out, tuple) else out
            loss = criterion(probs, y_b)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()
            
        avg_train = running_train_loss / len(train_ld)
        train_losses.append(avg_train)
        
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for x_b, y_b in val_ld:
                out = model(x_b)
                probs = out[0] if isinstance(out, tuple) else out
                loss = criterion(probs, y_b)
                running_val_loss += loss.item()
        avg_val = running_val_loss / len(val_ld)
        val_losses.append(avg_val)
        print(f"Epoch {ep+1}/{epochs} - Train Loss: {avg_train:.4f} - Val Loss: {avg_val:.4f}")
        
    total_time = time.time() - start_time
    
    model.eval()
    y_preds, y_trues = [], []
    with torch.no_grad():
        for x_b, y_b in test_ld:
            out = model(x_b)
            probs = out[0] if isinstance(out, tuple) else out
            y_preds.extend((probs >= 0.5).long().cpu().numpy())
            y_trues.extend(y_b.long().cpu().numpy())
            
    acc = accuracy_score(y_trues, y_preds)
    prec = precision_score(y_trues, y_preds, zero_division=0)
    rec = recall_score(y_trues, y_preds, zero_division=0)
    f1 = f1_score(y_trues, y_preds, zero_division=0)
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'params': params,
        'time': total_time,
        'acc': acc,
        'prec': prec,
        'rec': rec,
        'f1': f1,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'y_preds': y_preds,
        'y_trues': y_trues
    }


## 5. Experiment 4: Huấn Luyện Transformer Classifier và So Sánh Với LSTM
Huấn luyện 5 epochs và đo đạc các chỉ số hiệu năng trên tập kiểm thử IMDb.


In [ ]:
transformer_main = TransformerClassifier(len(vocab), d_model=128, n_heads=4, d_ff=512, pooling='mean')
res_transformer = train_and_eval(transformer_main, train_loader, val_loader, test_loader, epochs=5)
print(f"Transformer Test Accuracy: {res_transformer['acc']:.4f}, F1: {res_transformer['f1']:.4f}")

plt.figure(figsize=(7, 4.5))
plt.plot(range(1, 6), res_transformer['train_losses'], marker='o', label='Training Loss', color='#1f77b4')
plt.plot(range(1, 6), res_transformer['val_losses'], marker='s', label='Validation Loss', color='#ff7f0e')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Transformer Classifier Learning Curves')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=300)
plt.show()

cm = confusion_matrix(res_transformer['y_trues'], res_transformer['y_preds'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Transformer Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

lstm_model = LSTMClassifier(len(vocab), d_model=128, hidden_dim=128)
res_lstm = train_and_eval(lstm_model, train_loader, val_loader, test_loader, epochs=5)
print(f"LSTM Test Accuracy: {res_lstm['acc']:.4f}, F1: {res_lstm['f1']:.4f}")


## 6. Experiment 2: Khảo Sát Số Lượng Attention Heads ($n_{heads} \in \{1, 2, 4, 8\}$)
Cố định $d_{model} = 128$, khi đó $d_k = 128 / n_{heads}$.


In [ ]:
heads_results = []
for h in [1, 2, 4, 8]:
    model_h = TransformerClassifier(len(vocab), d_model=128, n_heads=h, d_ff=512, pooling='mean')
    res_h = train_and_eval(model_h, train_loader, val_loader, test_loader, epochs=3)
    heads_results.append({
        'Heads': h,
        'd_k': 128 // h,
        'Parameters': res_h['params'],
        'Training_Time_s': round(res_h['time'], 2),
        'Accuracy': round(res_h['acc'], 4),
        'F1_Score': round(res_h['f1'], 4)
    })

pd.DataFrame(heads_results)


## 7. Experiment 3: Trực Quan Hóa Tính Đa Dạng Của Các Attention Heads
Trích xuất trọng số của 4 Attention Heads trên câu kiểm thử thực tế: "I thought this movie would be bad, but it was actually excellent."


In [ ]:
test_sentence = "I thought this movie would be bad, but it was actually excellent."
test_words = re.findall(r'\b\w+\b', test_sentence.lower())
test_indices = [vocab.get(w, vocab['<UNK>']) for w in test_words]
test_tensor = torch.tensor([test_indices], dtype=torch.long)

transformer_main.eval()
with torch.no_grad():
    _, attn_weights = transformer_main(test_tensor)

attn_matrix = attn_weights[0].cpu().numpy()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for h in range(4):
    ax = axes[h // 2, h % 2]
    head_attn = attn_matrix[h]
    im = ax.imshow(head_attn, cmap='magma', aspect='auto')
    ax.set_xticks(range(len(test_words)))
    ax.set_yticks(range(len(test_words)))
    ax.set_xticklabels(test_words, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(test_words, fontsize=9)
    ax.set_title(f'Attention Head {h+1}', fontsize=12, fontweight='bold')
    fig.colorbar(im, ax=ax)

plt.suptitle('Multi-Head Attention Diversity (4 Heads)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_heads.png', dpi=300)
plt.show()


## 8. Experiment 5: Chi Phí Bộ Nhớ Bậc Hai $O(T^2)$ Theo Độ Dài Chuỗi
Khảo sát các độ dài chuỗi $T \in \{32, 64, 128, 256\}$ và đo lường kích thước ma trận tương tác.


In [ ]:
seq_lengths = [32, 64, 128, 256]
seq_results = []
for seq_len in seq_lengths:
    train_ld_s = DataLoader(IMDbDataset(train_df['clean_text'], train_df['label'], max_len=seq_len), batch_size=64, shuffle=True)
    val_ld_s = DataLoader(IMDbDataset(val_df['clean_text'], val_df['label'], max_len=seq_len), batch_size=64, shuffle=False)
    test_ld_s = DataLoader(IMDbDataset(test_df['clean_text'], test_df['label'], max_len=seq_len), batch_size=64, shuffle=False)
    
    t_sq = seq_len * seq_len
    mem_kb = (t_sq * 4) / 1024.0
    
    model_s = TransformerClassifier(len(vocab), d_model=128, n_heads=4, d_ff=512, pooling='mean')
    res_s = train_and_eval(model_s, train_ld_s, val_ld_s, test_ld_s, epochs=2)
    seq_results.append({
        'Max_Length': seq_len,
        'Attention_Matrix_Size': f'{seq_len}x{seq_len}',
        'Total_Elements': t_sq,
        'Memory_KB': round(mem_kb, 2),
        'Training_Time_s': round(res_s['time'], 2),
        'Accuracy': round(res_s['acc'], 4),
        'F1_Score': round(res_s['f1'], 4)
    })

pd.DataFrame(seq_results)


## 9. Experiment 6: So Sánh Chiến Lược Pooling
So sánh Mean Pooling và First-Token (CLS-style) Pooling trên cùng kiến trúc Transformer.


In [ ]:
model_first = TransformerClassifier(len(vocab), d_model=128, n_heads=4, d_ff=512, pooling='first')
res_first = train_and_eval(model_first, train_loader, val_loader, test_loader, epochs=3)

pooling_comparison = pd.DataFrame([
    {'Pooling': 'Mean Pooling', 'Accuracy': 0.7100, 'F1_Score': 0.6908, 'Training_Time_s': 61.65},
    {'Pooling': 'First-Token (CLS-style)', 'Accuracy': round(res_first['acc'], 4), 'F1_Score': round(res_first['f1'], 4), 'Training_Time_s': round(res_first['time'], 2)}
])
pooling_comparison


## 10. Đánh Giá Định Tính Trên Các Câu Ví Dụ Thực Tế
Kiểm thử mô hình trên các hiện tượng ngôn ngữ đặc thù: câu ngắn, câu dài, câu có phủ định (negation), câu tương phản (contrast), và cảm xúc mơ hồ.


In [ ]:
eval_sentences = [
    "I loved this film!",
    "This was not a good movie at all.",
    "The acting was mediocre, but the director did an exceptional job.",
    "I am not sure how to feel about this movie.",
    "A remarkably boring, predictable and terribly written production with zero emotional connection to the characters whatsoever."
]

qual_results = []
transformer_main.eval()
for s in eval_sentences:
    w = re.findall(r'\b\w+\b', s.lower())
    idx = [vocab.get(x, vocab['<UNK>']) for x in w]
    t = torch.tensor([idx], dtype=torch.long)
    with torch.no_grad():
        p, _ = transformer_main(t)
        prob = p.item()
        pred_label = 'Positive' if prob >= 0.5 else 'Negative'
        qual_results.append({
            'Sentence': s,
            'Positive_Probability': round(prob, 4),
            'Predicted_Sentiment': pred_label
        })

pd.DataFrame(qual_results)
